# PM versus correlated matching: measured decoder cost

Run this notebook in a **Colab CPU runtime**. It benchmarks standard PyMatching (PM) and PyMatching's two-pass correlated matching (CM) on identical experimental detector events from [Google Quantum AI's Zenodo archive](https://zenodo.org/records/6804040). No test split, trained router, or claimed router saving is involved.

The results measure **these PyMatching implementations on this Colab CPU**. Stored Google predictions are checked where available; matching them is necessary before describing the accuracy as a reproduction of the archived decoders. The archive stays in `/content`, outside Git.

## 1. Install and record the environment

PyMatching 2.3 or newer supports `enable_correlations`. Run both decoders in this same runtime; Colab CPU types can change between sessions.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'stim>=1.13', 'pymatching>=2.3', 'requests'], check=True)

In [ ]:
import hashlib
import itertools
import os
import platform
import shutil
import time
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import numpy as np
import pandas as pd
import pymatching
import stim

WORK = Path('/content/decoder_runtime')
WORK.mkdir(parents=True, exist_ok=True)
print({'python': sys.version.split()[0], 'stim': stim.__version__,
       'pymatching': pymatching.__version__, 'numpy': np.__version__,
       'platform': platform.platform(), 'processor': platform.processor(),
       'logical_cpu_count': os.cpu_count()})
CPU_MODEL = next((line.split(':', 1)[1].strip() for line in Path('/proc/cpuinfo').read_text().splitlines() if line.startswith('model name')), 'unknown')
print('CPU model:', CPU_MODEL)

## 2. Get and inspect the archive

The published ZIP is about 315 MB. The cell downloads it once and checks the MD5 shown on Zenodo. If downloading in Colab fails, upload the ZIP or mount Drive and set `ARCHIVE` to its location before running this cell. Internal filenames are discovered from the ZIP rather than assumed.

In [ ]:
import requests
ARCHIVE = WORK / 'google_qec3v5_experiment_data.zip'
URL = 'https://zenodo.org/records/6804040/files/google_qec3v5_experiment_data.zip?download=1'
EXPECTED_MD5 = 'a7fd8b481c3087090093106382dc217d'
if not ARCHIVE.exists():
    temporary = ARCHIVE.with_suffix('.partial')
    with requests.get(URL, stream=True, timeout=(30, 120)) as response:
        response.raise_for_status()
        with temporary.open('wb') as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
    temporary.replace(ARCHIVE)
digest = hashlib.md5()
with ARCHIVE.open('rb') as source:
    for chunk in iter(lambda: source.read(1024 * 1024), b''):
        digest.update(chunk)
assert digest.hexdigest() == EXPECTED_MD5, f'Archive checksum mismatch: {digest.hexdigest()}'
print('Verified archive:', ARCHIVE, 'bytes:', ARCHIVE.stat().st_size)

In [ ]:
with ZipFile(ARCHIVE) as archive:
    members = [name for name in archive.namelist() if not name.endswith('/')]
    readmes = [name for name in members if PurePosixPath(name).name.lower() == 'readme.txt']
    if readmes:
        print('Archive README:', readmes[0])
        print(archive.read(readmes[0]).decode('utf-8', errors='replace')[:8000])
by_dir = {}
for name in members:
    path = PurePosixPath(name)
    by_dir.setdefault(str(path.parent), {})[path.name] = name
required = {'circuit_detector_error_model.dem', 'detection_events.b8'}
candidates = sorted(directory for directory, files in by_dir.items() if required <= files.keys())
print('Configurations with a DEM and detector events:', len(candidates))
for index, directory in enumerate(candidates[:40]):
    print(index, directory)
assert candidates, 'No matching DEM/detection-event pair found. Inspect ZIP member names and its README.'

## 3. Select one surface-code configuration

Set `CONFIG_FILTER` to part of the desired experiment path (for example a distance, basis, or round count), then choose `CONFIG_INDEX` from the printed matches. Start with one configuration; do not pool different configurations into one timing result. Only its necessary files are extracted.

In [ ]:
CONFIG_FILTER = 'surface_code'
CONFIG_INDEX = 0
matches = [directory for directory in candidates if CONFIG_FILTER.lower() in directory.lower()]
print('Matching configurations:')
for index, directory in enumerate(matches[:40]):
    print(index, directory)
assert matches, 'No configuration matched. Inspect candidates and change CONFIG_FILTER.'
chosen_dir = matches[CONFIG_INDEX]
selected = by_dir[chosen_dir]
print('Selected:', chosen_dir)
print('Available files:', sorted(selected))
wanted = ['circuit_detector_error_model.dem', 'detection_events.b8',
          'obs_flips_actual.01', 'obs_flips_predicted_by_pymatching.01',
          'obs_flips_predicted_by_correlated_matching.01']
DEST = WORK / hashlib.sha256(chosen_dir.encode()).hexdigest()[:12]
DEST.mkdir(exist_ok=True)
paths = {}
with ZipFile(ARCHIVE) as archive:
    for basename in wanted:
        if basename in selected:
            destination = DEST / basename
            with archive.open(selected[basename]) as source, destination.open('wb') as output:
                shutil.copyfileobj(source, output)
            paths[basename] = destination
print('Extracted:', sorted(paths))

## 4. Parse and validate the same shots for both decoders

Stim's `b8` format packs bits in little-endian bit order within each byte. Its reader uses the DEM detector count; unpack only the selected first `N_SHOTS` rows. Prediction and outcome files, if present, are read in that same row order.

In [ ]:
N_SHOTS = 10_000  # Set to 1_000 for a pilot.
dem = stim.DetectorErrorModel.from_file(paths['circuit_detector_error_model.dem'])
assert dem.num_detectors > 0 and dem.num_observables > 0
packed = stim.read_shot_data_file(path=paths['detection_events.b8'], format='b8',
                                  num_detectors=dem.num_detectors, bit_packed=True)
assert packed.ndim == 2 and packed.shape[1] == (dem.num_detectors + 7) // 8
assert len(packed) >= N_SHOTS, f'Only {len(packed)} shots available; reduce N_SHOTS.'
shots = np.unpackbits(packed[:N_SHOTS], axis=1, bitorder='little')[:, :dem.num_detectors]
shots = np.ascontiguousarray(shots, dtype=np.uint8)
assert shots.shape == (N_SHOTS, dem.num_detectors)
print({'shots': len(shots), 'detectors': dem.num_detectors,
       'observables': dem.num_observables, 'DEM_correlation_separators': str(dem).count('^')})
if '^' not in str(dem):
    print('Check DEM content: no ^ separators were found, so correlation information may be absent.')

def read_01(basename):
    path = paths.get(basename)
    if path is None:
        return None
    with path.open('r', encoding='ascii') as source:
        lines = [line.strip() for line in itertools.islice(source, N_SHOTS)]
    assert len(lines) == N_SHOTS, f'{basename} has fewer than {N_SHOTS} rows'
    assert all(len(line) == dem.num_observables and set(line) <= {'0', '1'} for line in lines), basename
    return np.asarray([[int(bit) for bit in line] for line in lines], dtype=np.uint8)

actual = read_01('obs_flips_actual.01')
stored_pm = read_01('obs_flips_predicted_by_pymatching.01')
stored_cm = read_01('obs_flips_predicted_by_correlated_matching.01')
print('Reference files available:', {'actual': actual is not None, 'PM': stored_pm is not None, 'CM': stored_cm is not None})

## 5. Construct decoders and check predictions

Construction is excluded from per-shot timings. CM enables correlations both when constructing the matching graph and when decoding. A stored-prediction mismatch means this is a controlled comparison of the stated implementations, not an exact reproduction of the archive's decoders.

In [ ]:
build_start = time.perf_counter_ns()
pm = pymatching.Matching.from_detector_error_model(dem, enable_correlations=False)
pm_build_s = (time.perf_counter_ns() - build_start) / 1e9
build_start = time.perf_counter_ns()
cm = pymatching.Matching.from_detector_error_model(dem, enable_correlations=True)
cm_build_s = (time.perf_counter_ns() - build_start) / 1e9
print({'PM_build_seconds': pm_build_s, 'CM_build_seconds': cm_build_s})
assert pm.num_detectors == cm.num_detectors == dem.num_detectors

pm_pred = np.asarray(pm.decode_batch(shots, enable_correlations=False)).reshape(N_SHOTS, -1)
cm_pred = np.asarray(cm.decode_batch(shots, enable_correlations=True)).reshape(N_SHOTS, -1)
assert pm_pred.shape == cm_pred.shape == (N_SHOTS, dem.num_observables)
for name, predicted, reference in [('PM', pm_pred, stored_pm), ('CM', cm_pred, stored_cm)]:
    if reference is not None:
        mismatches = int(np.any(predicted != reference, axis=1).sum())
        print(f'{name} versus stored prediction: {mismatches}/{N_SHOTS} differing shots')
if actual is not None:
    for name, predicted in [('PM', pm_pred), ('CM', cm_pred)]:
        errors = int(np.any(predicted != actual, axis=1).sum())
        print(f'{name} logical error rate: {errors}/{N_SHOTS} = {errors/N_SHOTS:.4%}')

## 6. Time batch throughput

Both calls receive the same `shots` array. Warm-up and prediction checks above are outside the timer. Alternating call order reduces systematic ordering effects. Each timed run returns predictions; do not include DEM parsing, file reading, or decoder construction in these figures.

In [ ]:
REPEATS = 5
timing_rows = []
for repetition in range(REPEATS):
    order = [('PM', pm, False), ('CM', cm, True)]
    if repetition % 2:
        order.reverse()
    for name, decoder, correlated in order:
        start = time.perf_counter_ns()
        result = decoder.decode_batch(shots, enable_correlations=correlated)
        elapsed_ns = time.perf_counter_ns() - start
        assert len(result) == N_SHOTS
        timing_rows.append({'configuration': chosen_dir, 'mode': 'batch', 'decoder': name,
                            'repetition': repetition + 1, 'shots': N_SHOTS,
                            'elapsed_s': elapsed_ns / 1e9,
                            'us_per_shot': elapsed_ns / N_SHOTS / 1e3})
batch_runs = pd.DataFrame(timing_rows)
batch_summary = batch_runs.groupby('decoder')['us_per_shot'].agg(['median', 'min', 'max'])
display(batch_summary)
pm_batch_us = batch_summary.loc['PM', 'median']
cm_batch_us = batch_summary.loc['CM', 'median']
print(f'Batch CM/PM cost ratio: {cm_batch_us / pm_batch_us:.3f}x')
print(f'Batch CM minus PM: {cm_batch_us - pm_batch_us:.3f} µs/shot')
print(f'Median throughput: PM {1e6/pm_batch_us:,.0f}, CM {1e6/cm_batch_us:,.0f} shots/s')

## 7. Time sequential per-shot latency

This separately measures Python calls to `decode` one shot at a time on the same subset. It includes Python call overhead for **both** decoders. Use this mode when assessing a router that chooses and decodes each shot immediately; do not substitute batch throughput for online latency.

In [ ]:
SINGLE_SHOT_N = min(1_000, N_SHOTS)
single_shots = shots[:SINGLE_SHOT_N]
for decoder, correlated in [(pm, False), (cm, True)]:
    for row in single_shots[:min(100, SINGLE_SHOT_N)]:
        decoder.decode(row, enable_correlations=correlated)
single_rows = []
for repetition in range(REPEATS):
    order = [('PM', pm, False), ('CM', cm, True)]
    if repetition % 2:
        order.reverse()
    for name, decoder, correlated in order:
        start = time.perf_counter_ns()
        for row in single_shots:
            decoder.decode(row, enable_correlations=correlated)
        elapsed_ns = time.perf_counter_ns() - start
        single_rows.append({'configuration': chosen_dir, 'mode': 'sequential', 'decoder': name,
                            'repetition': repetition + 1, 'shots': SINGLE_SHOT_N,
                            'elapsed_s': elapsed_ns / 1e9,
                            'us_per_shot': elapsed_ns / SINGLE_SHOT_N / 1e3})
single_runs = pd.DataFrame(single_rows)
single_summary = single_runs.groupby('decoder')['us_per_shot'].agg(['median', 'min', 'max'])
display(single_summary)
pm_single_us = single_summary.loc['PM', 'median']
cm_single_us = single_summary.loc['CM', 'median']
print(f'Sequential CM/PM cost ratio: {cm_single_us / pm_single_us:.3f}x')
print(f'Sequential CM minus PM: {cm_single_us - pm_single_us:.3f} µs/shot')
runs = pd.concat([batch_runs, single_runs], ignore_index=True)
runs['stim_version'] = stim.__version__
runs['pymatching_version'] = pymatching.__version__
runs['cpu_model'] = CPU_MODEL
runs['archive_md5'] = EXPECTED_MD5
runs['dem_sha256'] = hashlib.sha256(paths['circuit_detector_error_model.dem'].read_bytes()).hexdigest()
output_path = WORK / 'decoder_timing_runs.csv'
runs.to_csv(output_path, index=False)
print('Saved repeated timing measurements:', output_path)

## Interpreting these results

The CM/PM ratio and difference above establish decoder cost **for this configuration and Colab CPU**. Repeat with other distances, bases, and round counts, recording each separately. Decoder construction times are reported but excluded from per-shot cost.

To claim router savings later, align the router's feature representation and decisions with these exact shots. Time feature extraction, model inference, and the selected decoder in the **same mode** as the always-CM baseline. Then compute `100 * (T_always_CM - T_routed) / T_always_CM`. The earlier 44.1% CM usage rate is only an illustrative estimate until decisions on these shots are available.